# Stage 12: Final face-off on held-out questions

Four systems answer every eval question with **Claude Sonnet 5.5 through the API**, with the same prompt, schema,
`get_image` tool and citation checks. Every answer is graded by **Claude Opus 5.5 through the API**.

| system | context given to the model |
|---|---|
| **RAG, top 10** | dense + BM25 fused, top 30 reranked by MiniLM, best 10 kept |
| **RAG, top 15** | the same, best 15 kept (added during the face-off, before any of its answers) |
| **RAG, dynamic, no cap** | top 50 fused reranked by MiniLM; keep every candidate scoring at least -3.68 (at least 3, no upper limit) |
| **Whole handbook** | all 254 articles, ~673k Claude tokens, prompt-cached between questions |

**Questions** (`evals/handbook_eval.jsonl`):
- **60 held-out questions.** These are the headline. They were written after all tuning, and were committed before
  any system answered them. 10 of them are questions the handbook doesn't cover.
- **40 dev questions.** These were used to choose methods, so their scores are optimistic; they are shown for
  reference.

**What was frozen first** (checked below from the git history):
- the held-out questions;
- the no-cap configuration (`evals/frozen_dynamic_nocap.json`).

Runs: `python -m handbook_qa.compare answer|judge rag_top10 rag_top15 rag_dynamic_nocap full_context`. RAG top 10 and the
whole handbook had already answered the 40 dev questions with the same code, model and prompt (notebook 11), so the
runner only added the 60 held-out ones.

In [1]:
import collections
import subprocess

import numpy as np
import pandas as pd

from handbook_qa import compare
from handbook_qa import evaluation as ev
from handbook_qa.config import ROOT, load_settings
from handbook_qa.judge import SCORES
from handbook_qa.retrieval import DynamicSelection, load_corpus

settings = load_settings()
SYSTEMS = {"rag_top10": "RAG, top 10", "rag_top15": "RAG, top 15", "rag_dynamic_nocap": "RAG, dynamic no cap",
           "full_context": "Whole handbook"}
PRICES = {"claude-sonnet-5-5": (2.0, 10.0), "claude-opus-5-5": (4.0, 20.0)}  # $ per M tokens, in / out
pd.set_option("display.max_colwidth", 110)

questions = ev.load_eval_set(compare.EVAL_SET)
ev.resolve(questions, load_corpus(settings.paths.chunks, settings.paths.image_index))
by_id = {q.id: q for q in questions}
print(collections.Counter((q.split, q.type) for q in questions))
print("no-cap system:", DynamicSelection.load(compare.FROZEN_NOCAP))

Counter({('holdout', 'multi-article'): 15, ('holdout', 'single'): 14, ('dev', 'multi-article'): 12, ('holdout', 'reworded'): 12, ('dev', 'existing'): 11, ('holdout', 'not-covered'): 10, ('dev', 'reworded'): 9, ('holdout', 'multi-section'): 9, ('dev', 'multi-section'): 8})
no-cap system: DynamicSelection(cutoff=-3.68, candidates=50, min_keep=3, max_keep=50, model='Xenova/ms-marco-MiniLM-L-6-v2')


## What was frozen, and when

In [2]:
def git_times(*args) -> list[int]:
    out = subprocess.run(["git", "log", "--format=%ct", *args], cwd=ROOT, capture_output=True, text=True).stdout
    return sorted(int(t) for t in out.split())


holdout_added = git_times("-S", '"split": "holdout"', "--", "evals/handbook_eval.jsonl")[0]
nocap_frozen = git_times("--diff-filter=A", "--", "evals/frozen_dynamic_nocap.json")[0]
answers_committed = {s: git_times("--", str(compare.answers_path(s).relative_to(ROOT))) for s in SYSTEMS}
holdout_ids = {q.id for q in questions if q.split == "holdout"}
for s, times in answers_committed.items():
    # The commit that first contains held-out answers must come after the held-out questions were committed.
    later = [t for t in times if t > holdout_added]
    assert later, f"{s}: no answers committed after the held-out questions"
    if s == "rag_dynamic_nocap":
        assert min(times) > nocap_frozen, "no-cap answers predate its frozen config"
print("held-out questions committed:", holdout_added, "| no-cap config frozen:", nocap_frozen)
print("OK: the held-out questions and the no-cap config were committed before any answers using them")

held-out questions committed: 1792987564 | no-cap config frozen: 1792987324
OK: the held-out questions and the no-cap config were committed before any answers using them


## Load answers and judgements

In [3]:
answers = {s: compare.read_rows(compare.answers_path(s)) for s in SYSTEMS}
judged = {s: compare.read_rows(compare.judgements_path(s)) for s in SYSTEMS}
for s, name in SYSTEMS.items():
    print(f"{name:22} answers {len(answers[s])}, judgements {len(judged[s])}, sources "
          f"{dict(collections.Counter(r['source'] for r in answers[s].values()))}")


def cost(usage: dict, model: str) -> float:
    price_in, price_out = PRICES[model]
    return (usage.get("input_tokens", 0) * price_in + usage.get("cache_creation_input_tokens", 0) * price_in * 1.25
            + usage.get("cache_read_input_tokens", 0) * price_in * 0.1 + usage.get("output_tokens", 0) * price_out) / 1e6


rows = []
for s, name in SYSTEMS.items():
    for qid, a in answers[s].items():
        q, j = by_id[qid], judged[s].get(qid)
        recall = (ev.chunk_recall(a["excerpt_ids"], q, len(a["excerpt_ids"]))
                  if s != "full_context" else (1.0 if q.covered else np.nan))
        rows.append({
            "system": name, "id": qid, "split": q.split, "type": q.type, "covered": q.covered,
            "coverage label": a["coverage"], "excerpts": len(a["excerpt_ids"]) if s != "full_context" else 254,
            "chunk recall": recall,
            "input tokens": a["usage"]["input_tokens"] + a["usage"]["cache_creation_input_tokens"] + a["usage"]["cache_read_input_tokens"],
            "answer $": cost(a["usage"], settings.answer.model), "seconds": a["seconds"],
            "citations ok": a["citations_ok"], "citations": a["citations_total"],
            **({dim: SCORES[dim][j[dim]] for dim in SCORES} if j else {}),
            "fully right": (j["correctness"] == "correct" and j["completeness"] == "complete"
                            and j["groundedness"] == "grounded" and j["coverage_honesty"] == "appropriate") if j else np.nan,
            "judge $": cost(j["usage"], settings.judge.model) if j else np.nan,
        })
df = pd.DataFrame(rows)
order = list(SYSTEMS.values())


def summary(sub: pd.DataFrame) -> pd.DataFrame:
    g = sub.groupby("system", sort=False)
    out = pd.DataFrame({
        "questions": g.size(),
        "chunk recall (answerable)": g["chunk recall"].mean(),
        "mean excerpts": g["excerpts"].mean(),
        "citations correct": g["citations ok"].sum() / g["citations"].sum(),
        "correctness": g["correctness"].mean(), "completeness": g["completeness"].mean(),
        "groundedness": g["groundedness"].mean(), "coverage honesty": g["coverage_honesty"].mean(),
        "fully right": g["fully right"].mean(),
        "mean input tokens": g["input tokens"].mean(), "median seconds": g["seconds"].median(),
        "answer $ (total)": g["answer $"].sum(), "answer $ / question": g["answer $"].mean(),
    })
    return out.loc[[n for n in order if n in out.index]]

RAG, top 10            answers 100, judgements 100, sources {'anthropic-api:anthropic:claude-sonnet-5-5': 100}
RAG, top 15            answers 100, judgements 100, sources {'anthropic-api:anthropic:claude-sonnet-5-5': 100}
RAG, dynamic no cap    answers 100, judgements 100, sources {'anthropic-api:anthropic:claude-sonnet-5-5': 100}
Whole handbook         answers 100, judgements 100, sources {'anthropic-api:anthropic:claude-sonnet-5-5': 100}


## Headline: the 60 held-out questions

In [4]:
holdout = df[df["split"] == "holdout"]
summary(holdout).T.round(3)

system,"RAG, top 10","RAG, top 15","RAG, dynamic no cap",Whole handbook
questions,60.000,60.000,60.000,60.000
chunk recall (answerable),0.803,0.853,0.807,1.000
mean excerpts,10.000,15.000,10.433,254.000
citations correct,1.000,1.000,0.995,0.984
correctness,0.983,0.983,0.975,1.000
completeness,0.867,0.875,0.867,0.975
groundedness,0.967,0.958,0.975,0.908
coverage honesty,0.983,1.000,0.983,0.967
fully right,0.750,0.767,0.767,0.750
mean input tokens,10429.800,14885.217,10944.350,673279.950


### Answerable (50) vs not covered (10)

In [5]:
summary(holdout[holdout["covered"]]).T.round(3)

system,"RAG, top 10","RAG, top 15","RAG, dynamic no cap",Whole handbook
questions,50.000,50.000,50.000,50.000
chunk recall (answerable),0.803,0.853,0.807,1.000
mean excerpts,10.000,15.000,11.200,254.000
citations correct,1.000,1.000,0.995,0.983
correctness,0.980,0.980,0.970,1.000
completeness,0.840,0.850,0.840,0.970
groundedness,0.970,0.950,0.970,0.920
coverage honesty,0.980,1.000,0.980,0.960
fully right,0.720,0.720,0.720,0.760
mean input tokens,10483.340,14941.740,11646.440,673281.260


In [6]:
nc = holdout[~holdout["covered"]]
pd.DataFrame({
    "said 'not covered'": nc.groupby("system", sort=False)["coverage label"].apply(lambda c: (c == "not_covered").sum()),
    "said 'partial'": nc.groupby("system", sort=False)["coverage label"].apply(lambda c: (c == "partial").sum()),
    "answered as 'full'": nc.groupby("system", sort=False)["coverage label"].apply(lambda c: (c == "full").sum()),
    "judge: correct": nc.groupby("system", sort=False)["correctness"].apply(lambda c: (c == 1).sum()),
    "judge: honest coverage": nc.groupby("system", sort=False)["coverage_honesty"].apply(lambda c: (c == 1).sum()),
}).loc[order]

,said 'not covered',said 'partial',answered as 'full',judge: correct,judge: honest coverage
system,,,,,
"RAG, top 10",9,1,0,10,10
"RAG, top 15",9,1,0,10,10
"RAG, dynamic no cap",8,2,0,10,10
Whole handbook,7,3,0,10,10


### By question type (held out; completeness, 1 = complete)

In [7]:
holdout.pivot_table(index="type", columns="system", values="completeness", aggfunc="mean")[order] \
    .assign(questions=holdout[holdout["system"] == order[0]].groupby("type").size()).round(2)

system,"RAG, top 10","RAG, top 15","RAG, dynamic no cap",Whole handbook,questions
type,,,,,
multi-article,0.90,0.83,0.80,0.93,15
multi-section,0.78,0.89,0.89,1.00,9
not-covered,1.00,1.00,1.00,1.00,10
reworded,0.67,0.67,0.67,0.96,12
single,0.96,1.00,1.00,1.00,14


### Are the differences real? Paired bootstrap over the held-out questions

For each pair of systems: the difference in the share of fully right answers (and in completeness), with a 95%
interval from resampling questions. An interval that includes 0 means 60 questions can't separate the two.

In [8]:
rng = np.random.default_rng(0)
wide = holdout.pivot_table(index="id", columns="system", values=["fully right", "completeness"])
pairs = [("RAG, top 10", "Whole handbook"), ("RAG, top 10", "RAG, top 15"), ("RAG, top 10", "RAG, dynamic no cap"),
         ("RAG, top 15", "Whole handbook"), ("RAG, dynamic no cap", "Whole handbook")]
rows_ci = []
for metric in ["fully right", "completeness"]:
    for a, b in pairs:
        diff = (wide[(metric, b)] - wide[(metric, a)]).astype(float).to_numpy()
        boots = [rng.choice(diff, len(diff)).mean() for _ in range(10000)]
        rows_ci.append({"metric": metric, "comparison": f"{b} minus {a}", "difference": diff.mean(),
                        "95% low": np.percentile(boots, 2.5), "95% high": np.percentile(boots, 97.5)})
pd.DataFrame(rows_ci).round(3)

,metric,comparison,difference,95% low,95% high
0,fully right,"Whole handbook minus RAG, top 10",0.000,-0.150,0.150
1,fully right,"RAG, top 15 minus RAG, top 10",0.017,-0.050,0.083
2,fully right,"RAG, dynamic no cap minus RAG, top 10",0.017,-0.083,0.117
3,fully right,"Whole handbook minus RAG, top 15",-0.017,-0.150,0.117
4,fully right,"Whole handbook minus RAG, dynamic no cap",-0.017,-0.167,0.133
5,completeness,"Whole handbook minus RAG, top 10",0.108,0.042,0.183
6,completeness,"RAG, top 15 minus RAG, top 10",0.008,-0.025,0.042
7,completeness,"RAG, dynamic no cap minus RAG, top 10",0.000,-0.042,0.042
8,completeness,"Whole handbook minus RAG, top 15",0.100,0.033,0.175
9,completeness,"Whole handbook minus RAG, dynamic no cap",0.108,0.042,0.183


## For reference: the 40 dev questions and all 100

In [9]:
pd.concat({"dev (40)": summary(df[df["split"] == "dev"])[["questions", "chunk recall (answerable)", "completeness",
                                                          "groundedness", "fully right", "answer $ / question"]],
           "all (100)": summary(df)[["questions", "chunk recall (answerable)", "completeness", "groundedness",
                                     "fully right", "answer $ / question"]]}).round(3)

questions  chunk recall (answerable)  \
          system                                                      
dev (40)  RAG, top 10                 40                      0.846   
          RAG, top 15                 40                      0.879   
          RAG, dynamic no cap         40                      0.862   
          Whole handbook              40                      1.000   
all (100) RAG, top 10                100                      0.822   
          RAG, top 15                100                      0.865   
          RAG, dynamic no cap        100                      0.831   
          Whole handbook             100                      1.000   

                               completeness  groundedness  fully right  \
          system                                                         
dev (40)  RAG, top 10                 0.862         0.962        0.725   
          RAG, top 15                 0.850         0.962        0.700   
          RAG, dynamic no cap         0.875         0.975        0.750   
          Whole handbook              0.938         0.900        0.650   
all (100) RAG, top 10                 0.865         0.965        0.740   
          RAG, top 15                 0.865         0.960        0.740   
          RAG, dynamic no cap         0.870         0.975        0.760   
          Whole handbook              0.960         0.905        0.710   

                               answer $ / question  
          system                                    
dev (40)  RAG, top 10                        0.024  
          RAG, top 15                        0.033  
          RAG, dynamic no cap                0.030  
          Whole handbook                     0.183  
all (100) RAG, top 10                        0.023  
          RAG, top 15                        0.032  
          RAG, dynamic no cap                0.026  
          Whole handbook                     0.173

## Cost of the face-off

In [10]:
spend = df.groupby("system", sort=False)[["answer $", "judge $"]].sum().loc[order]
spend.loc["total"] = spend.sum()
spend.round(2)

,answer $,judge $
system,,
"RAG, top 10",2.34,4.66
"RAG, top 15",3.23,6.39
"RAG, dynamic no cap",2.61,5.23
Whole handbook,17.26,3.50
total,25.43,19.77


## Where the systems disagree (held out)

In [11]:
verdict = holdout.assign(v=lambda d: np.where(d["fully right"] == 1, "fully right",
                                              d["coverage label"] + " / c" + d["completeness"].astype(str)))
per_q = verdict.pivot_table(index="id", columns="system", values="v", aggfunc="first")[order]
per_q.insert(0, "type", [by_id[i].type for i in per_q.index])
per_q.insert(1, "question", [by_id[i].question for i in per_q.index])
differ = per_q[per_q[order].nunique(axis=1) > 1]
print(len(differ), "of", len(per_q), "held-out questions get different outcomes across systems")
differ

29 of 60 held-out questions get different outcomes across systems


system,type,question,"RAG, top 10","RAG, top 15","RAG, dynamic no cap",Whole handbook
id,,,,,,
A-14,single,Will my Weave stats be used to decide whether I get let go?,fully right,fully right,fully right,full / c1.0
A-16,reworded,A teammate really went above and beyond for me this week. What's the easiest way to give them a public sho...,not_covered / c0.0,not_covered / c0.0,not_covered / c0.0,fully right
A-17,reworded,Is it okay to share my views on an upcoming election in our work Slack channels?,not_covered / c0.0,not_covered / c0.0,not_covered / c0.0,full / c0.5
A-20,multi-section,"I've been picked as a new joiner's onboarding buddy. What's expected of me, and when are their probation c...",fully right,full / c1.0,fully right,fully right
A-21,multi-section,"I have a complaint about a colleague that we couldn't sort out informally. How do I raise it formally, and...",full / c0.5,fully right,partial / c0.5,fully right
A-22,multi-article,"I'm planning to leave PostHog to start my own company. What happens to my vested share options, and might ...",partial / c0.5,partial / c0.0,partial / c0.5,fully right
A-25,not-covered,"If I refer a friend and they get hired, do I get a referral bonus?",fully right,fully right,fully right,not_covered / c1.0
A-27,not-covered,Does PostHog offer a paid sabbatical once you've been there a few years?,fully right,fully right,fully right,not_covered / c1.0
B-12,single,I'm a new engineer and need access to PostHog's AWS account. How do I get it?,full / c1.0,full / c1.0,full / c1.0,fully right


The not-covered questions, side by side:

In [12]:
for q in [q for q in questions if q.split == "holdout" and not q.covered][:4]:
    print(f"=== {q.id}: {q.question}\nreference: {q.answer}\n")
    for s, name in SYSTEMS.items():
        a = answers[s][q.id]
        print(f"--- {name} [{a['coverage']}]: {a['answer'][:400]}\n")

=== A-25: If I refer a friend and they get hired, do I get a referral bonus?
reference: The handbook doesn't cover this. It mentions referrals only as a source of hires (most hires apply directly or through referrals, and the engineering hiring page points to the careers page in case you want to refer someone), but says nothing about a bonus or reward for referring.

--- RAG, top 10 [not_covered]: The handbook excerpts I have don't cover this. The engineering hiring page says you can find open roles on the careers page "in case you want to refer someone" [1]. It doesn't say whether a referral bonus exists or how it works.

--- RAG, top 15 [not_covered]: The handbook excerpts I have don't cover this. They don't say whether you get a referral bonus if a friend you refer is hired. The engineering hiring page says you can find open roles on the careers page "in case you want to refer someone", but it says nothing about a bonus [1].

--- RAG, dynamic no cap [not_covered]: The handbook excer

## Reading the results (written after the run)

- **Fully right answers are a tie.** On the 60 held-out questions, top 10, top 15, no-cap and the whole
  handbook score 75-77%. The 95% intervals for every difference include zero.
- **The whole handbook is more complete but less grounded.** Its completeness is 0.975 against 0.867 for top 10,
  and the interval for that difference excludes zero. The gain is largest on reworded questions (0.96 against 0.67),
  which retrieval often misses. But it adds more small unsupported details (groundedness 0.91 against
  0.96-0.98), so its share of fully right answers ends up level with RAG. It is also about 7 times as expensive per
  question ($0.17 against $0.02) and twice as slow.
- **Not-covered questions are handled well by all four.** Every system declined all 10, and none answered one as if
  it were covered. The whole handbook more often labelled these "partial", because it mentioned nearby facts.
- **More context barely helps RAG.** Top 15 finds more of the evidence (chunk recall 0.85 against 0.80), but its
  answers barely change, and it costs about 40% more. The no-cap dynamic system behaves like top 10.
- **Recommendation:** keep RAG top 10 as the default. It's the cheapest and fastest, and as good as anything here
  on fully right answers. Its weak spot is reworded questions, where retrieval misses the passage. The whole-handbook
  option is worth offering when completeness matters more than cost.